# Part 9 · Notebook 05 — PCA residual statistical arbitrage

**Sessions:** S5 (PCA / residual statistical arbitrage) · [Lesson plan](../../docs/lessons/PART_09_STATISTICAL_TRADING.md) · graded labs in [`labs/part09/`](../../labs/part09/)

**You will:**
1. Extract statistical factors with PCA.
2. Turn each stock's cumulative residual into an Avellaneda–Lee s-score.
3. Write the open and close rules.
4. Backtest a dollar-neutral residual book, with and without residual reversion in the data.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with known parameters (true half-lives, true pairs, true hedge ratios, planted premia), so every estimator can be checked against the truth.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p9lib.py is in notebooks/part09/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p9lib as p

p.use_course_style()

## 1. From pairs to portfolios

A pair hedges one stock with another. Residual stat-arb hedges each stock with a few **common factors** and trades what is left. `p.factor_universe()` has 60 stocks driven by 3 factors; for half of them (`reverting`), the idiosyncratic price has a small OU component with a 3-day half-life, hidden under daily noise a little larger than its own moves. The other half is pure noise. PCA of the correlation matrix finds the factors:

In [ ]:
R, reverting = p.factor_universe()
ev = np.linalg.eigvalsh(np.corrcoef(R.to_numpy().T))[::-1]
share = ev / ev.sum()
print(f"{R.shape[1]} stocks × {R.shape[0]} days; variance explained by the first 5 components: {np.round(share[:5], 3)}")
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(1, 16), share[:15]); ax.set_title("scree plot: one market factor, two small ones, then noise"); plt.show()

## 2. Residuals and the s-score

Over the last 60 days: standardize the returns, take the top `k` eigenvectors as **eigenportfolios**, regress each stock's raw returns on `[1, factors]`, and cumulate the residuals into a price-like series `X`. This part is done for you:

In [ ]:
window, k = 60, 3
Rw = R.iloc[-window:]
Z = ((Rw - Rw.mean()) / Rw.std()).to_numpy()
_, vec = np.linalg.eigh(np.corrcoef(Z.T))                  # eigenvalues in ascending order
factors = Z @ vec[:, ::-1][:, :k]                          # returns of the top-k eigenportfolios
A = np.c_[np.ones(window), factors]
coef, *_ = np.linalg.lstsq(A, Rw.to_numpy(), rcond=None)
X = pd.DataFrame(np.cumsum(Rw.to_numpy() - A @ coef, axis=0), columns=R.columns)   # cumulative residuals
X.iloc[:, :6].plot(figsize=(10, 3), lw=1, title="cumulative residuals X of six stocks (60 days)"); plt.show()

Now the per-stock step. Fit `X` as an AR(1) (`b, a`); skip the stock if it doesn't revert, or reverts too slowly. The OU equilibrium is `μ = a/(1 − b)` with standard deviation `σ_eq = √(var(AR residuals, ddof=2) / (1 − b²))`, and the **s-score** is `(X_last − μ)/σ_eq`: how many equilibrium standard deviations the stock is away from its fair level relative to the factors.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def s_score(X, max_half_life=None):
    X = np.asarray(X, dtype=float)
    b, a = np.polyfit(X[:-1], X[1:], 1)                   # AR(1): X[t+1] = a + b·X[t] + e
    if not 0 < b < 1 or (max_half_life is not None and np.log(2) / -np.log(b) > max_half_life):
        return np.nan                                     # not reverting (fast enough): no s-score
    resid = X[1:] - (a + b * X[:-1])
    mu = ...                                              # ✍️ the equilibrium level
    sig_eq = ...                                          # ✍️ the equilibrium std
    return float((X[-1] - mu) / sig_eq)

mine = {mh: pd.Series({c: p.attempt(s_score, X[c], mh) for c in X.columns}) for mh in (None, 10)}
mine = p.check("s_score", mine, {mh: p.s_scores(R, k, window, mh).reindex(R.columns) for mh in (None, 10)})
s = mine[10].dropna()
print(f"{mine[None].notna().sum()} stocks have an AR(1) slope in (0, 1); {len(s)} also have a half-life of 10 days or less")
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.bar(s.index, s.to_numpy(), color=np.where(s.abs() > 1.25, "#eb6834", "#8a8984"))
ax.axhline(1.25, color="#eb6834", ls="--", lw=0.8); ax.axhline(-1.25, color="#eb6834", ls="--", lw=0.8)
ax.tick_params(axis="x", labelrotation=90, labelsize=7); ax.set_title("s-scores today: beyond ±1.25 opens a trade"); plt.show()

## 3. Trading rules (Avellaneda & Lee, 2010)

Per stock, given yesterday's position: **flat** → buy (`+1`) if `s < −1.25`, sell (`−1`) if `s > 1.25`; **long** → close once `s > −0.5`; **short** → close once `s < 0.75`. A stock with no s-score today is closed. Return a Series in the order of `prev`.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def s_score_positions(s, prev, open_=1.25, close_long=0.5, close_short=0.75):
    out = {}
    for name, cur in prev.items():
        v = s.get(name, np.nan)
        if np.isnan(v):
            out[name] = 0.0                               # no s-score today (not reverting): close
        elif cur == 0:
            out[name] = ...                               # ✍️ open long below −open_, short above +open_, else flat
        elif cur > 0:
            out[name] = ...                               # ✍️ close the long once v > −close_long
        else:
            out[name] = 0.0 if v < close_short else -1.0
    return pd.Series(out, dtype=float)

toy_s = pd.Series({"A": -1.5, "B": 1.4, "C": -0.3, "D": 0.6, "E": 0.9, "F": -0.8})
toy_prev = pd.Series({"A": 0.0, "B": 0.0, "C": 1.0, "D": -1.0, "E": -1.0, "F": 1.0, "G": 1.0})
cases = [(toy_s, toy_prev), (s, pd.Series(0.0, index=R.columns))]
mine = [p.attempt(s_score_positions, *c) for c in cases]
mine = p.check("s_score_positions", mine, [p.s_score_positions(*c) for c in cases])
display(pd.DataFrame({"s": toy_s, "yesterday": toy_prev, "today": mine[0]}).T)
print(f"today, from flat: {int((mine[1] > 0).sum())} longs and {int((mine[1] < 0).sum())} shorts")

## 4. The residual book

`p.residual_backtest` runs this every day: s-scores from the last 60 days (3 factors, half-life ≤ 10), positions by your rules, **dollar-neutral** weights (+0.5 spread over the longs, −0.5 over the shorts), earned the next day, 5 bp per unit of turnover. Compare with a universe where **no** residual reverts.

In [ ]:
bt = p.residual_backtest(R)
bt_free = p.residual_backtest(R, cost_bps=0)
R0, _ = p.factor_universe(reversion_share=0.0)
bt_none = p.residual_backtest(R0)
market = R.mean(axis=1).loc[bt["returns"].index]
pd.DataFrame({"Sharpe": [p.sharpe(bt["returns"]), p.sharpe(bt_free["returns"]), p.sharpe(bt_none["returns"])],
              "correlation with the market": [np.corrcoef(bt["returns"], market)[0, 1], np.nan, np.nan]},
             index=["half the residuals revert, 5 bp", "half revert, no costs", "none revert, 5 bp"]).round(3)

In [ ]:
pos = bt["positions"].abs().to_numpy()
print(f"on average {(bt['positions'] > 0).sum(axis=1).mean():.1f} longs and {(bt['positions'] < 0).sum(axis=1).mean():.1f} shorts; "
      f"{pos[:, reverting].sum() / pos.sum():.0%} of the position-days are in the reverting half")
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.plot(bt["returns"].cumsum(), label="reverting residuals"); ax.plot(bt_none["returns"].cumsum(), label="no reversion")
ax.legend(); ax.set_title("residual stat-arb book, net of costs"); plt.show()

The book earns a Sharpe of about 1 net of costs with essentially zero market correlation, and loses its costs when there is nothing to find. Note the second line: the positions are barely tilted toward the stocks that really revert. Sixty days can't tell a 3-day OU component from noise stock by stock; the edge comes from averaging many weak bets. That is also why costs and crowding (August 2007) matter so much for this strategy.

## Wrap-up

* Hedge each stock with factors, trade the residual; the book is close to market- and factor-neutral by construction.
* The s-score is an OU z-score of the cumulative residual; trade only fast reverters.
* Many small bets, high turnover: the cost assumption decides the result.
* Graded version: `labs/part09/week32_statarb` (`s_scores`, `s_score_positions`, `residual_backtest`).